<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/test(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm

# =====================
# Config
# =====================
BATCH_SIZE   = 128
LR           = 0.001
EPOCHS       = 50
WEIGHT_DECAY = 1e-4

train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test"

CKPT_DIR = "/content/sample_data/CNN_model_CIFAR10"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [73]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
    T.RandomErasing(p=0.1)
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616])
])


In [74]:
class CIFARFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(root))

        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                self.samples.append((os.path.join(folder_path, img_name), label_idx))

        print(f"Found {len(self.samples)} images in {root}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = imageio.imread(img_path)

        if self.transform:
            img = self.transform(img)

        return img, label


In [75]:
train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path,  transform=test_transform)


Found 50001 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train.
Found 10000 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test.


In [76]:
train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

# ⭐ FIX: shuffle=True so evaluation varies each run
test_dl = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=True,   # <--- THIS IS THE FIX
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)


In [77]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [78]:
model = BetterCNN().to(device)
model = model.to(memory_format=torch.channels_last)
model = torch.compile(model)

opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")


In [79]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_dl):.4f}")


In [80]:
def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted', zero_division=0)
    rec = recall_score(labels_all, preds_all, average='weighted', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return f1


In [ ]:
resume_path = os.path.join(CKPT_DIR, "last_epoch.pth")
start_epoch = 0

opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

if os.path.exists(resume_path):
    print("Resuming from last checkpoint...")
    checkpoint = torch.load(resume_path)

    loaded_state_dict = checkpoint["model"]
    new_state_dict = {}
    for k, v in loaded_state_dict.items():
        if k.startswith("_orig_mod."):
            new_key = k[len("_orig_mod."):]
        else:
            new_key = k
        new_state_dict[new_key] = v

    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(new_state_dict)

    opt.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler = torch.amp.GradScaler("cuda")

    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed at epoch {start_epoch}")


In [83]:
best_f1 = 0.0

for epoch in range(start_epoch, EPOCHS):
    train_one_epoch(epoch)
    scheduler.step()

    model_to_save_state = model
    if hasattr(model, '_orig_mod'):
        model_to_save_state = model._orig_mod

    torch.save({
        "epoch": epoch,
        "model": model_to_save_state.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict()
    }, resume_path)

    print(f"Checkpoint saved: {resume_path}")


Epoch 2 | Loss: 1.5745
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 3 | Loss: 1.2536
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 4 | Loss: 1.1130
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 5 | Loss: 1.0212
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 6 | Loss: 0.9703
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


...

Epoch 8 | Loss: 0.8834
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 9 | Loss: 0.8561
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 10 | Loss: 0.8330
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 11 | Loss: 0.7972
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


KeyboardInterrupt: 

In [ ]:
print("\n=== Running evaluation 5 times with shuffle=True ===\n")
for i in range(10):
    print(f"Run {i+1}:")
    evaluate()
    print()



=== Running evaluation 5 times with shuffle=True ===

Run 1:
Accuracy:  0.7673
Precision: 0.7682
Recall:    0.7673
F1 Score:  0.7653

Run 2:
Accuracy:  0.7673
Precision: 0.7682
Recall:    0.7673
F1 Score:  0.7653

Run 3:
Accuracy:  0.7673
Precision: 0.7682
Recall:    0.7673
F1 Score:  0.7653

Run 4:
Accuracy:  0.7673
Precision: 0.7682
Recall:    0.7673
F1 Score:  0.7653

Run 5:
